In [12]:
#Program to access Camera of UR5e's Robotiq arm
#Written by Rahul Antony

In [14]:
pip install opencv-python

Note: you may need to restart the kernel to use updated packages.


In [15]:
import cv2
print(cv2.__version__)

4.10.0


In [16]:
import requests
import cv2
import numpy as np

In [17]:
# Replace with your robot's IP address
ROBOT_IP = '172.22.22.2'
CAMERA_URL = f'http://{ROBOT_IP}:4242/current.jpg?annotations=<on|off>'  # Example URL to access camera image
#Before running the code, you can test the access of camera by pasting it into a web browser (http://172.22.22.2:4242/current.jpg?annotations=<on|off>)

In [18]:
def get_camera_image():
    """
    Fetches an image from the robot's camera and returns it as a NumPy array.
    """
    try:
        # Send an HTTP GET request to fetch the camera image
        response = requests.get(CAMERA_URL)
        
        # Check if the request was successful
        if response.status_code == 200:
            # Convert the response content to a NumPy array
            image = np.array(bytearray(response.content), dtype=np.uint8)
            image = cv2.imdecode(image, cv2.IMREAD_COLOR)  # Decode the image
            return image
        else:
            print(f"Failed to retrieve image. Status code: {response.status_code}")
            return None
    except requests.exceptions.RequestException as e:
        print(f"Error accessing camera: {e}")
        return None

In [19]:
def main():
    print("Press 'q' for stopping the program")
    while True:
        # Get the current camera image
        image = get_camera_image()

        if image is not None:
            # Display the image in a window
            cv2.imshow('UR5e Camera Feed', image)
        
        # Exit if 'q' is pressed
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

    # Convert the image to grayscale
    gray_image = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)

    # Apply Gaussian Blur to reduce noise and improve edge detection
    blurred_image = cv2.GaussianBlur(gray_image, (5, 5), 0)

    # Perform edge detection using the Canny algorithm
    edges = cv2.Canny(blurred_image, threshold1=50, threshold2=150)

    # Find contours in the edged image
    contours, _ = cv2.findContours(edges, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    # Sort contours by area and keep the largest one (which should be the pallet)
    contours = sorted(contours, key=cv2.contourArea, reverse=True)
    if len(contours) > 0:
        pallet_contour = contours[0]  # Assuming the pallet is the largest contour
    else:
        print("No contours found.")
        exit()

    # Draw the contour on the original image
    cv2.drawContours(image, [pallet_contour], -1, (0, 255, 0), 2)  # Green contour line

    # Display the original image with the detected pallet boundary
    cv2.imshow("Pallet Boundary", image)


    # Clean up
    cv2.destroyAllWindows()

if __name__ == "__main__":
    main()


Press 'q' for stopping the program
